In [ ]:
import time
from dataclasses import dataclass
from typing import Dict, Tuple, List

import dimod
from neal import SimulatedAnnealingSampler

DEC_WEIGHTS = (1, 2, 3, 3)  # SBE digit weights

def sbe_affine_bits(var: str, J: int, L=0.0, U=1.0) -> Tuple[float, Dict[str, float]]:
    """
    Returns an affine representation:
        x = const + sum_i alpha_i * z_i
    using SBE decimal digits + tail bit.
    """
    const = L
    scale = (U - L)

    coeffs: Dict[str, float] = {}
    for j in range(1, J + 1):
        place = 10 ** (-j)
        for k, w in enumerate(DEC_WEIGHTS, start=1):
            b = f"z_{var}_{j}_{k}"
            coeffs[b] = coeffs.get(b, 0.0) + scale * place * w

    tail = f"z_{var}_tail_J{J}"
    coeffs[tail] = coeffs.get(tail, 0.0) + scale * (10 ** (-J))
    return const, coeffs

def add_square_of_affine(Qlin, Qquad, offset_ref, c0: float, coeffs: Dict[str, float], weight: float = 1.0):
    """
    Add: weight * (c0 + sum_i a_i z_i)^2 to QUBO.
    Uses z_i^2 = z_i.
    """
    # constant
    offset_ref[0] += weight * (c0 * c0)

    items = list(coeffs.items())

    # linear terms: weight*(a_i^2 + 2*c0*a_i)*z_i
    for bi, ai in items:
        Qlin[bi] = Qlin.get(bi, 0.0) + weight * (ai * ai + 2.0 * c0 * ai)

    # quadratic terms: weight*(2*a_i*a_j)*z_i*z_j
    for i in range(len(items)):
        bi, ai = items[i]
        for j in range(i + 1, len(items)):
            bj, aj = items[j]
            u, v = (bi, bj) if bi < bj else (bj, bi)
            Qquad[(u, v)] = Qquad.get((u, v), 0.0) + weight * (2.0 * ai * aj)

def build_bqm(Qlin, Qquad, offset: float) -> dimod.BinaryQuadraticModel:
    return dimod.BinaryQuadraticModel(Qlin, Qquad, offset, vartype=dimod.BINARY)

def qubo_stats(bqm) -> Dict[str, int]:
    return {
        "n_vars": len(bqm.variables),
        "n_quadratic": len(bqm.quadratic),
        "n_linear": len(bqm.linear),
    }

def decode_from_affine(sample: Dict[str, int], const: float, coeffs: Dict[str, float]) -> float:
    val = const
    for b, a in coeffs.items():
        val += a * float(sample.get(b, 0))
    return val


def solve_example2_rolling_with_slack(
    J0=2, Jstep=2, Jmax=8,
    penalty_lambda=100.0,
    num_reads=200, sweeps=2500, seed=13,
):
    sampler = SimulatedAnnealingSampler()

    def solve_at(J1, J2):
        # Choose slack precision tied to current grid
        Js = max(J1, J2)

        Qlin, Qquad = {}, {}
        offset = [0.0]

        c1, a1 = sbe_affine_bits("x1", J1, 0.0, 1.0)
        c2, a2 = sbe_affine_bits("x2", J2, 0.0, 1.0)
        cs, as_ = sbe_affine_bits("s",  Js, 0.0, 1.0)

        # objective: (x1-0.3)^2 + (x2-0.6)^2
        add_square_of_affine(Qlin, Qquad, offset, c1 - 0.3, a1, weight=1.0)
        add_square_of_affine(Qlin, Qquad, offset, c2 - 0.6, a2, weight=1.0)

        # constraint penalty: lambda*(x1+x2+s-1)^2
        # g = (c1+c2+cs-1) + sum(a1+a2+as) z
        g0 = (c1 + c2 + cs - 1.0)
        gcoeffs = {}
        for d in (a1, a2, as_):
            for b, a in d.items():
                gcoeffs[b] = gcoeffs.get(b, 0.0) + a

        add_square_of_affine(Qlin, Qquad, offset, g0, gcoeffs, weight=penalty_lambda)

        bqm = build_bqm(Qlin, Qquad, offset[0])

        t0 = time.perf_counter()
        ss = sampler.sample(bqm, num_reads=num_reads, sweeps=sweeps, seed=seed)
        t1 = time.perf_counter()

        best = ss.first.sample
        x1 = decode_from_affine(best, c1, a1)
        x2 = decode_from_affine(best, c2, a2)
        s  = decode_from_affine(best, cs, as_)

        obj = (x1 - 0.3) ** 2 + (x2 - 0.6) ** 2
        resid = (x1 + x2 + s - 1.0)  # should be ~0 if penalty dominates
        infeas = max(0.0, x1 + x2 - 1.0)  # the real inequality violation

        merit = obj + penalty_lambda * (resid ** 2)

        return {
            "J": (J1, J2, Js),
            "bqm": bqm,
            "time_s": (t1 - t0),
            "x": (x1, x2, s),
            "obj": obj,
            "resid": resid,
            "infeas": infeas,
            "merit": merit,
            "stats": qubo_stats(bqm),
        }

    # Rolling strategy: refine x1 then x2 greedily 
    history = []
    J1 = J2 = J0
    best = solve_at(J1, J2)
    history.append(best)

    improved = True
    while improved and (J1 < Jmax or J2 < Jmax):
        improved = False

        # Try refine x1
        if J1 + Jstep <= Jmax:
            cand = solve_at(J1 + Jstep, J2)
            if cand["merit"] < best["merit"] - 1e-10:
                J1 += Jstep
                best = cand
                history.append(best)
                improved = True
                continue

        # Try refine x2
        if J2 + Jstep <= Jmax:
            cand = solve_at(J1, J2 + Jstep)
            if cand["merit"] < best["merit"] - 1e-10:
                J2 += Jstep
                best = cand
                history.append(best)
                improved = True
                continue

        # If neither helps, stop

    # Monolithic at (Jmax,Jmax)
    mono = solve_at(Jmax, Jmax)
    return history, mono

def print_report2(history, mono):
    print("\n--- Rolling precision history (constrained) ---")
    for k, r in enumerate(history):
        J1, J2, Js = r["J"]
        x1, x2, s = r["x"]
        print(
            f"it={k:02d}  J=(x1:{J1},x2:{J2},s:{Js})  "
            f"obj={r['obj']:.3e}  resid={r['resid']:+.2e}  infeas={r['infeas']:.2e}  "
            f"x1={x1:.6f}  x2={x2:.6f}  s={s:.6f}  "
            f"nvars={r['stats']['n_vars']}  nquad={r['stats']['n_quadratic']}  time={r['time_s']:.3f}s"
        )

    print("\n--- Monolithic (constrained) ---")
    J1, J2, Js = mono["J"]
    x1, x2, s = mono["x"]
    print(
        f"J=(x1:{J1},x2:{J2},s:{Js})  obj={mono['obj']:.3e}  resid={mono['resid']:+.2e}  infeas={mono['infeas']:.2e}  "
        f"x1={x1:.6f}  x2={x2:.6f}  s={s:.6f}  "
        f"nvars={mono['stats']['n_vars']}  nquad={mono['stats']['n_quadratic']}  time={mono['time_s']:.3f}s"
    )

if __name__ == "__main__":
    hist, mono = solve_example2_rolling_with_slack()
    print_report2(hist, mono)



--- Rolling precision history (constrained) ---
it=00  J=(x1:2,x2:2,s:2)  obj=1.000e-03  resid=+2.22e-16  infeas=0.00e+00  x1=0.310000  x2=0.630000  s=0.060000  nvars=27  nquad=351  time=0.078s
it=01  J=(x1:4,x2:2,s:4)  obj=2.560e-06  resid=+2.22e-16  infeas=0.00e+00  x1=0.301600  x2=0.600000  s=0.098400  nvars=43  nquad=903  time=0.114s
it=02  J=(x1:6,x2:2,s:6)  obj=1.541e-32  resid=+2.22e-16  infeas=0.00e+00  x1=0.300000  x2=0.600000  s=0.100000  nvars=59  nquad=1711  time=0.172s

--- Monolithic (constrained) ---
J=(x1:8,x2:8,s:8)  obj=1.040e-04  resid=+2.22e-16  infeas=0.00e+00  x1=0.302000  x2=0.610000  s=0.088000  nvars=99  nquad=4851  time=0.462s
